# KGRID interactive data explorer
Choose **one WMU device**, enter **UTC start/end times**, and press **Download and plot**. Data downloads, interactive plots display, and matching **PNG** and offline **HTML** files save automatically.

**Up to and including one hour:** waveform + available Parquet measurements. **More than one hour:** Parquet only; no `.raw` or `.7z` downloads. Parquet keeps its native cadence (currently about 2 sps).

Run all cells. Network access occurs only on **Download and plot** or optional **Preview source files**. **Load local dataset** and **Offline synthetic demo** work without credentials.

Setup: run `uv sync`, then `uv run kgrid-notebook` from the repository root. Select **KGRID Analysis (project Python)** and use **Kernel → Restart Kernel and Run All Cells** after updates; saved widget output from an old session is not live. Credentials belong in `~/.config/kestrelgrid/credentials.json`, never in notebook cells. See `README.md`.

In [ ]:
from pathlib import Path

from IPython.display import display

from kestrelgrid_tools.explorer import Explorer

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'config/wmu_locations.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the kestrelgrid_analysis checkout')
explorer = Explorer(ROOT)
display(explorer.widget)

## How to use

1. Choose a WMU and defined UTC interval. Default device: **4021**; the prefilled recent minute is editable. Duration/data mode updates immediately.
2. Press **Download and plot**. Discovery and serial downloads run automatically, then PNG + HTML save with device/window filenames and clickable links. **Preview source files** is optional. Transfer budgets are under **Advanced settings**.

**PJM July 22 test presets:** select WMU **4012 (Charlottesville, closest listed to Dominion/PJM)** or **4011 (Nashville, Eastern comparison)**, then choose the event hour (07:30–08:30 EDT / 11:30–12:30 UTC) or full Eastern day (04:00 UTC July 22–04:00 UTC July 23). The presentation gives 07:56:09 for line opening; Eastern timezone is an explicit assumption. Both devices and both windows were tested interactively.
3. Adjust local plot bounds/traces, then **Update plots (local)**. Blank bounds show all downloaded coverage. For waveform cycles, choose 0.1–1 second; only that portion is loaded from full-resolution disk storage.
4. Zoom, pan and toggle traces. **Export current view** saves the modified view. The initial download always saves both formats.

A one-hour waveform at 4,000 samples/s has 14.4 million samples (~27.5 MiB raw payload). Data streams to disk and decodes in ~250,000-sample chunks. A transfer estimate is checked first; high-rate units or compressed daily archives may need a larger budget.

The HTML embeds Plotly.js and the plotted samples, so it works offline. Python/widget controls remain notebook-only. Zooming a reduced min/max overview does not reconstruct omitted samples. Statistics use full-resolution data. PNG export requires Chrome/Chromium for Kaleido.

## Timing, quality and frequency

All selections and plot labels use UTC. Live inspection on 2026-11-05 found that daily filenames correspond to **America/Los_Angeles** partitions; the nominal 2026-11-04 raw archive ends around 2026-11-05 08:00 UTC. DST-aware partition conversion is implemented. Actual recording timestamps are retained in the manifest.

Current Parquet columns are `t` (epoch seconds), `rms`, `theta`, and `gps`, at approximately 2 sps. New RMS/frequency schemas are supported explicitly; unknown schemas raise an error. Raw sampling rate comes from each frame header. GPS-equipped does not imply every frame is GPS-locked.

**Frequency plots are automatic:** stored Parquet frequency first, otherwise a full-resolution waveform estimate (interpolated positive zero crossings and one-second cycle-count regression at 2 estimates/s), otherwise theta converted using a verified per-WMU convention. Sources and quality are recorded in plot titles and manifests.

**WMU 4021 is verified:** radians, 60 Hz reference, positive derivative. Its cached hour agrees with waveform estimates at 0.9993 correlation and ~0.00052 Hz RMSE at measurement timestamps. Evidence is in `config/frequency_conventions.json`. Other devices are checked against simultaneous waveform data before conventions are reused; references are not guessed.

For radians, `f = f_ref + sign * d(theta)/dt / (2*pi)`. At 2 sps deviations beyond ±1 Hz can alias undetectably. The waveform estimator is not a PMU compliance claim. Advanced settings allow a known manual override. Existing cached datasets gain frequency locally, without downloading.

This server is bandwidth constrained. The notebook will not mirror it. More than 12 waveform archives per selection is blocked; coordinate larger downloads with the provider. Existing local bundles can be reloaded with no network access.

## Access the selected data in Python

Use `explorer.dataset.waveforms['4021']` (disk-backed WaveformStore), `explorer.dataset.measurements['4021']` and `explorer.dataset.frequencies['4021']` (DataFrames), `explorer.dataset.summary`, and `explorer.dataset.manifest`. Full waveform samples stay on disk; plots may use display reduction.

Optional examples (run manually after loading):
```python
dataset = explorer.dataset
display(dataset.summary)
store = dataset.waveforms['4021']
detail = store.to_dataframe(start='2026-11-05T07:59:10Z', end='2026-11-05T07:59:11Z')
display(detail.head())
detail.to_csv(dataset.directory / 'selected_voltage.csv', index=False)
# For full-resolution large exports, iterate store.iter_tables() in bounded batches.
```

For direct public-package CSV conversion of a frame-aligned raw sample:
```python
from kestrelgrid_tools.interpretation import export_raw_csv
raw_path = next(dataset.directory.glob('*.raw'))
export_raw_csv(raw_path, dataset.directory / 'frame_aligned_voltage.csv')
```
The direct raw export includes full boundary frames; normalized storage reflects the requested `[start, end)` interval. Repeated/backward raw timestamps are flagged, overlapping samples excluded in stream order, and gaps preserved. Raw CSV export may reject those source anomalies; export cleaned normalized batches instead. Clear notebook outputs before sharing or committing.